In [0]:
staging_dbo_pre_hchbar = dbutils.widgets.get("staging_dbo_pre_hchbar")
staging_dbo_hchbar = dbutils.widgets.get("staging_dbo_hchbar")
fact_accountsreceivable = dbutils.widgets.get("fact_accountsreceivable")
hchb_temp_dbo= dbutils.widgets.get("hchb_temp_dbo")
hchb_officemapping = dbutils.widgets.get("hchb_officemapping")
client = dbutils.widgets.get("client")
client_episodes = dbutils.widgets.get("client_episodes")
payerdimension = dbutils.widgets.get("payerdimension")
office = dbutils.widgets.get("office")
cubeserviceofficetxnsourcesystem = dbutils.widgets.get("cubeserviceofficetxnsourcesystem")
cubeserviceofficetxnweekendingdate = dbutils.widgets.get("cubeserviceofficetxnweekendingdate")
cubeserviceofficetxnservicedate = dbutils.widgets.get("cubeserviceofficetxnservicedate")
date = dbutils.widgets.get("date")
bears_oblist = dbutils.widgets.get("bears_oblist")
cubhub_alpha_claims = dbutils.widgets.get("cubhub_alpha_claims")
staging_dbo_hchb_bkp=dbutils.widgets.get("staging_dbo_hchb_bkp")
fact_accountsreceivable_bkp=dbutils.widgets.get("fact_accountsreceivable_bkp")

In [0]:
spark.sql(f"""
TRUNCATE TABLE {staging_dbo_pre_hchbar}
""")


In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW Results AS
SELECT
reporting_week_ending_date   AS reporting_week_ending_date,
source_system                AS source_system,
pps                          AS pps,
key_date                     AS key_date,
end_date                     AS end_date,
paid                         AS paid,
client_name                  AS client_name,
payor_type                   AS payor_type,
psid                         AS psid,
payor_source                 AS payor_source,
bill_date                    AS bill_date,
reporting_branch_code        AS reporting_branch_code,
revenue                      AS revenue,
earned_rev                   AS earned_rev,
adjustments                  AS adjustments,
unearned_rev                 AS unearned_rev,
cash                         AS cash,
credits                      AS credits,
refunds                      AS refunds,
eoe_to_qe_days               AS eoe_to_qe_days,
eoe_to_me_days               AS eoe_to_me_days,
episode_in_progress_amt      AS episode_in_progress_amt,
ar_days_0_30                 AS ar_days_0_30,
ar_days_31_60                AS ar_days_31_60,
ar_days_61_90                AS ar_days_61_90,
ar_days_91_120               AS ar_days_91_120,
ar_days_121_plus             AS ar_days_121_plus,
gross_ar                     AS gross_ar,
net_earned_ar                AS net_earned_ar

FROM {hchb_temp_dbo}
""")

# Verify the view was created successfully
print("Results view created successfully")
result_count = spark.sql("SELECT COUNT(*) as count FROM Results").collect()[0]['count']
print(f"Total records in Results view: {result_count}")

In [0]:
spark.sql(f"""
INSERT INTO {staging_dbo_pre_hchbar} (
    reporting_week_ending_date,
    source_system,
    pps,
    key_date,
    end_date,
    paid,
    client_name,
    payor_type,
    psid,
    payor_source,
    bill_date,
    reporting_branch_code,
    revenue,
    earned_rev,
    adjustments,
    unearned_rev,
    cash,
    credits,
    refunds,
    eoe_to_qe,
    eoe_to_me,
    episode_in_progress,
    days_0_30,
    days_31_60,
    days_61_90,
    days_91_120,
    days_121_plus,
    gross_ar,
    net_earned_ar
)
WITH numbered_rows AS (
    SELECT
        reporting_week_ending_date,
        source_system,
        pps,
        key_date,
        end_date,
        paid,
        client_name,
        payor_type,
        psid,
        payor_source,
        bill_date,
        reporting_branch_code,
        revenue,
        earned_rev,
        adjustments,
        unearned_rev,
        cash,
        credits,
        refunds,
        eoe_to_qe_days,
        eoe_to_me_days,
        episode_in_progress_amt,
        ar_days_0_30,
        ar_days_31_60,
        ar_days_61_90,
        ar_days_91_120,
        ar_days_121_plus,
        gross_ar,
        net_earned_ar,
        ROW_NUMBER() OVER (
            PARTITION BY
                reporting_week_ending_date,
                source_system,
                pps,
                key_date,
                end_date,
                paid,
                client_name,
                payor_type,
                psid,
                payor_source,
                bill_date,
                reporting_branch_code,
                revenue,
                earned_rev,
                adjustments,
                unearned_rev,
                cash,
                credits,
                refunds,
                eoe_to_qe_days,
                eoe_to_me_days,
                episode_in_progress_amt,
                ar_days_0_30,
                ar_days_31_60,
                ar_days_61_90,
                ar_days_91_120,
                ar_days_121_plus,
                gross_ar,
                net_earned_ar
            ORDER BY reporting_week_ending_date
        ) AS row_num
    FROM results
)
SELECT
    reporting_week_ending_date,
    source_system,
    pps,
    key_date,
    end_date,
    paid,
    client_name,
    payor_type,
    psid,
    payor_source,
    bill_date,
    reporting_branch_code,
    revenue,
    earned_rev,
    adjustments,
    unearned_rev,
    cash,
    credits,
    refunds,
    eoe_to_qe_days AS eoe_to_qe,
    eoe_to_me_days AS eoe_to_me,
    episode_in_progress_amt AS episode_in_progress,
    ar_days_0_30 AS days_0_30,
    ar_days_31_60 AS days_31_60,
    ar_days_61_90 AS days_61_90,
    ar_days_91_120 AS days_91_120,
    ar_days_121_plus AS days_121_plus,
    gross_ar,
    net_earned_ar
FROM numbered_rows
WHERE row_num = 1
""")

In [0]:
count = spark.sql(f"SELECT COUNT(*) as cnt FROM {staging_dbo_hchbar}").collect()[0]['cnt']
if count == 0 :
    print(f"Full load for {staging_dbo_hchbar} executed")
    spark.sql(f"""
    INSERT INTO {staging_dbo_hchbar} (
    reporting_week_ending_date,
    source_system,
    pps,
    key_date,
    end_date,
    paid,
    client_name,
    payor_type,
    psid,
    payor_source,
    bill_date,
    reporting_branch_code,
    revenue,
    earned_rev,
    adjustments,
    unearned_rev,
    cash,
    credits,
    refunds,
    eoe_to_qe,
    eoe_to_me,
    episode_in_progress,
    days_0_30,
    days_31_60,
    days_61_90,
    days_91_120,
    days_121_plus,
    gross_ar,
    net_earned_ar
    )
    SELECT
    CAST(ReportingWeekEndingDate AS date) as reporting_week_ending_date,
    CAST(SourceSystem AS string) as source_system ,
    CAST(PPS AS smallint) as pps,
    CAST(KeyDate AS date) as key_date,
    CAST(EndDate AS date) as end_date,
    CAST(PAID AS int) as paid,
    CAST(ClientName AS STRING) as client_name,
    CAST(Payor_Type AS STRING) as payor_type,
    CAST(psid AS int) as psid,
    CAST(Payor_Source AS STRING) as payor_source,
    CAST(BillDate AS date) as bill_date,
    CAST(Reporting_BranchCode AS STRING) as reporting_branch_code,
    CAST(Revenue AS decimal(15,4)) as revenue,
    CAST(EarnedRev AS decimal(15,4)) as earned_rev,
    CAST(Adjustments AS decimal(15,4)) as adjustments,
    CAST(UnEarnedRev AS decimal(15,4)) as unearned_rev,
    CAST(Cash AS decimal(15,4)) as cash,
    CAST(Credits AS decimal(15,4)) as credits,
    CAST(Refunds AS decimal(15,4)) as refunds,
    CAST(EOEtoQE AS int) as eoe_to_qe,
    CAST(EOEtoME AS int) as eoe_to_me,
    CAST(EpisodeInProgress AS decimal(15,4)) as episode_in_progress,
    CAST(0___30_Days AS decimal(15,4)) as days_0_30,
    CAST(31___60_Days AS decimal(15,4)) as days_31_60,
    CAST(61___90_Days AS decimal(15,4)) as days_61_90,
    CAST(91___120_Days AS decimal(15,4)) as days_91_120,
    CAST(121__Days AS decimal(15,4)) as days_121_plus,
    CAST(GrossAR AS decimal(15,4)) as gross_ar,
    CAST(NetEarnedAR AS decimal(15,4)) as net_earned_ar
    FROM {staging_dbo_hchb_bkp};
    """)
else:
    print("Skipping the full load")
 

In [0]:
spark.sql(f"""
INSERT INTO {staging_dbo_hchbar}
SELECT *
FROM {staging_dbo_pre_hchbar}
""")

In [0]:
spark.sql(f"""
MERGE INTO {staging_dbo_hchbar} AS ha
USING (
    SELECT DISTINCT
        ha.reporting_branch_code AS SourceCode,
        om.TargetOfficeNumber,
        ha.reporting_week_ending_date
    FROM {staging_dbo_hchbar} ha
    LEFT JOIN {hchb_officemapping} om
        ON ha.reporting_branch_code = om.SourceOfficeCode
    WHERE ha.reporting_branch_code RLIKE '[A-Z]'
      AND ha.reporting_week_ending_date = (
          SELECT MAX(reporting_week_ending_date)
          FROM {staging_dbo_hchbar}
      )
) mapping
ON ha.reporting_branch_code = mapping.SourceCode
AND ha.reporting_week_ending_date = mapping.reporting_week_ending_date
WHEN MATCHED THEN
UPDATE SET ha.reporting_branch_code = mapping.TargetOfficeNumber
""")

alpha_branch_count = spark.sql(f"""
    SELECT COUNT(*) as count
    FROM {staging_dbo_hchbar}
    WHERE reporting_branch_code RLIKE '[A-Z]'
    AND reporting_week_ending_date = (
        SELECT MAX(reporting_week_ending_date)
        FROM {staging_dbo_hchbar}
    )
""").collect()[0]['count']

In [0]:
count = spark.sql(f"SELECT COUNT(*) as cnt FROM {fact_accountsreceivable}").collect()[0]['cnt']
if count == 0 :
    print(f"Full load for {fact_accountsreceivable} executed")
    spark.sql(f"""
    INSERT INTO {fact_accountsreceivable} (
    reporting_week_ending_date_key,
    start_date_key,
    end_date_key,
    source_system_key,
    office_key,
    payor_key,
    client_key,
    invoice_balance_type_key,
    age_from_todays_date,
    age_from_quarter_end_date,
    invoice_number,
    balance,
    episode_in_progress,
    prorated_ar_balance,
    gross_ar,
    loaded_ts
    )
    SELECT
    CAST(Reporting_Week_Ending_Date_Key AS int) as reporting_week_ending_date_key ,
    CAST(Start_Date_Key AS int) as start_date_key,
    CAST(End_Date_Key AS int) as end_date_key,
    CAST(Source_System_Key AS int) as source_system_key,
    CAST(Office_Key AS int) as office_key,
    CAST(Payor_Key AS int) as payor_key,
    CAST(Client_Key AS int) as client_key,
    CAST(Invoice_Balance_Type_Key AS int) as invoice_balance_type_key,
    CAST(Age_From_Today_s_Date AS int) as age_from_todays_date,
    CAST(Age_From_Quarter_End_Date AS int) as age_from_quarter_end_date,
    CAST(Invoice_Number AS STRING) as invoice_number,
    CAST(Balance AS decimal(15,4)) as balance,
    CAST(Episode_In_Progress AS decimal(15,4)) as episode_in_progress,
    CAST(Prorated_AR_Balance AS decimal(15,4)) as prorated_ar_balance,
    CAST(Gross_AR AS decimal(15,4)) as gross_ar,
    current_timestamp() as loaded_ts
    FROM {fact_accountsreceivable_bkp};
    """)
else:
    print("Skipping the full load")
 

In [0]:
spark.sql("DROP VIEW IF EXISTS tmp_balance_type")

spark.sql(f"""
INSERT INTO {fact_accountsreceivable} (
    invoice_balance_type_key,
    reporting_week_ending_date_key,
    source_system_key,
    payor_key,
    client_key,
    office_key,
    start_date_key,
    end_date_key,
    balance,
    age_from_todays_date,
    age_from_quarter_end_date,
    invoice_number,
    episode_in_progress,
    gross_ar
)
SELECT 
    CASE WHEN Balance > 0 THEN 1 ELSE 2 END AS invoice_balance_type_key,
    W.WeekEndingDateKey AS reporting_week_ending_date_key,
    S.SourceSystemKey AS Source_System_Key,
    p.PayerKey AS payor_key,
    csi.ClientKey AS client_key,
    o.OfficeKey AS office_key,
    sd.ServiceDateKey AS start_date_key,
    sa.ServiceDateKey AS end_date_key,
    Balance as balance,
    AgefromMonthEnd AS age_from_todays_date,
    AgefromQuarterEnd AS age_from_quarter_end_date,
    NULL AS invoice_number,
    EpisodeInProgress AS episode_in_progress,
    GrossAR AS gross_ar
FROM (
    SELECT
        reporting_week_ending_date,
        source_system,
        pps,
        key_date,
        end_date,
        paid,
        psid,
        reporting_branch_code,
        net_earned_ar AS Balance,
        eoe_to_qe AS AgefromMonthEnd,
        eoe_to_me AS AgefromQuarterEnd,
        hcea.epi_id,
        episode_in_progress as EpisodeInProgress,
        a.gross_ar AS GrossAR
    FROM {staging_dbo_hchbar} a
    LEFT JOIN (
        SELECT epi_id, epi_paid
        FROM (
            SELECT 
                epi_id,
                epi_paid,
                ROW_NUMBER() OVER (PARTITION BY epi_paid ORDER BY epi_paid, epi_id) AS rnk
            FROM {client_episodes}
        )
        WHERE rnk = 1
    ) hcea
        ON a.paid = CAST(hcea.epi_paid AS STRING)
    WHERE reporting_week_ending_date = (
        SELECT MAX(reporting_week_ending_date)
        FROM {staging_dbo_hchbar}
    )
) AR
LEFT JOIN (
    SELECT sourcesystemID, clientkey, SourceSystem
    FROM (
        SELECT 
            sourcesystemID,
            clientkey,
            SourceSystem,
            ROW_NUMBER() OVER (PARTITION BY sourcesystemID, SourceSystem ORDER BY sourcesystemID) AS rnk1
        FROM {client}
    )
    WHERE rnk1 = 1
) csi 
    ON CAST(AR.epi_id AS STRING) = TRIM(CAST(csi.SourceSystemID AS STRING))
   AND csi.SourceSystem = 'HCHB'
LEFT JOIN {payerdimension} p
    ON CAST(AR.psid AS STRING) = p.PayerID
LEFT JOIN {office} o
    ON o.OfficeNumber = ABS(AR.reporting_branch_code)
LEFT JOIN {cubeserviceofficetxnsourcesystem} S
    ON S.SourceSystemName = AR.source_system
LEFT JOIN {cubeserviceofficetxnweekendingdate} W
    ON W.WeekEndingDate = AR.reporting_week_ending_date
LEFT JOIN {cubeserviceofficetxnservicedate} sd
    ON sd.ServiceDate = CAST(AR.key_date AS DATE)
LEFT JOIN {cubeserviceofficetxnservicedate} sa
    ON sa.ServiceDate = CAST(AR.end_date AS DATE)
""")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW max_reporting_week_key AS
SELECT REPLACE(MAX(reporting_week_ending_date), '-', '') AS max_week_key
FROM {staging_dbo_pre_hchbar}
""")

max_week = spark.sql("SELECT max_week_key FROM max_reporting_week_key").collect()[0]['max_week_key']

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW tmp_balance_type AS
SELECT 
    client_key,
    payor_key,
    SUM(balance) AS Blc,
    CASE 
        WHEN SUM(balance) > 0 THEN 1
        WHEN SUM(balance) < 0 THEN 2
        ELSE 0
    END AS New_Invoice_Balance_Type_Key
FROM {fact_accountsreceivable} ar
CROSS JOIN max_reporting_week_key m
WHERE ar.reporting_week_ending_date_key = m.max_week_key
  AND ar.source_system_key = 6
  AND ar.start_date_key IS NULL
GROUP BY 1,2
""")

spark.sql(f"""
MERGE INTO {fact_accountsreceivable} AS ar
USING (
    SELECT 
        client_key,
        payor_key,
        New_Invoice_Balance_Type_Key as invoice_balance_type_key,
        max_week_key as reporting_week_ending_date_key
    FROM (
        SELECT 
            tmp.client_key,
            tmp.payor_key,
            tmp.New_Invoice_Balance_Type_Key,
            m.max_week_key,
            ROW_NUMBER() OVER (PARTITION BY tmp.client_key, tmp.payor_key ORDER BY tmp.client_key) AS rn
        FROM tmp_balance_type tmp
        CROSS JOIN max_reporting_week_key m
    )
    WHERE rn = 1
) AS src
ON ar.client_key = src.client_key
   AND ar.payor_key = src.payor_key
   AND ar.reporting_week_ending_date_key = src.reporting_week_ending_date_key
   AND ar.source_system_key = 6
   AND ar.start_date_key IS NULL
WHEN MATCHED THEN
    UPDATE SET ar.invoice_balance_type_key = src.invoice_balance_type_key
""")


spark.sql(f"""
CREATE OR REPLACE TEMP VIEW calc_prorated AS
SELECT
    ar.client_key,
    ar.payor_key,
    ar.reporting_week_ending_date_key,
    ar.start_date_key,
    ar.source_system_key,
    CASE
        WHEN ar.source_system_key = 6
         AND p.Name LIKE '%%PDGM%%'
         AND ar.start_date_key IS NOT NULL
        THEN
            CASE
                WHEN date_add(
                    TO_DATE(CAST(ar.start_date_key AS STRING), 'yyyyMMdd'), 29
                ) < TO_DATE(CAST(ar.reporting_week_ending_date_key AS STRING), 'yyyyMMdd')
                THEN CAST(ar.balance AS DECIMAL(15,4))
                ELSE ROUND(
                    (
                        DATEDIFF(
                            TO_DATE(CAST(ar.reporting_week_ending_date_key AS STRING), 'yyyyMMdd'),
                            TO_DATE(CAST(ar.start_date_key AS STRING), 'yyyyMMdd')
                        ) + 1
                    ) / 30.0 * COALESCE(ar.gross_ar, 0),
                    4
                )
            END
        ELSE CAST(ar.balance AS DECIMAL(15,4))
    END AS Prorated_AR_Balance
FROM {fact_accountsreceivable} ar
CROSS JOIN max_reporting_week_key m
LEFT JOIN (
    SELECT PayerKey, MAX(Name) AS Name
    FROM {payerdimension}
    GROUP BY PayerKey
) p
    ON ar.payor_key = p.PayerKey
WHERE ar.reporting_week_ending_date_key = m.max_week_key
  AND ar.source_system_key = 6
""")

spark.sql(f"""
MERGE INTO {fact_accountsreceivable} ar
USING (
    SELECT 
        client_key,
        payor_key,
        reporting_week_ending_date_key,
        start_date_key,
        source_system_key,
        Prorated_AR_Balance as prorated_ar_balance
    FROM (
        SELECT 
            client_key,
            payor_key,
            reporting_week_ending_date_key,
            start_date_key,
            source_system_key,
            Prorated_AR_Balance,
            ROW_NUMBER() OVER (
                PARTITION BY client_key, payor_key, reporting_week_ending_date_key, start_date_key, source_system_key 
                ORDER BY client_key
            ) AS rn
        FROM calc_prorated
    )
    WHERE rn = 1
) c
ON ar.client_key = c.client_key
   AND ar.payor_key = c.payor_key
   AND ar.reporting_week_ending_date_key = c.reporting_week_ending_date_key
   AND ar.start_date_key = c.start_date_key
   AND ar.source_system_key = c.source_system_key
WHEN MATCHED THEN
    UPDATE SET ar.prorated_ar_balance = c.prorated_ar_balance
""")

summary = spark.sql(f"""
SELECT 
    'Accounts Receivable Fact Table Updated Successfully' AS Status,
    COUNT(*) AS TotalRecords,
    SUM(balance) AS TotalBalance,
    SUM(prorated_ar_balance) AS TotalProratedBalance,
    (SELECT max_week_key FROM max_reporting_week_key) AS ReportingWeek
FROM {fact_accountsreceivable} ar
CROSS JOIN max_reporting_week_key m
WHERE ar.reporting_week_ending_date_key = m.max_week_key
""")

summary.show(truncate=False)

In [0]:
quarter_end_result = spark.sql(f"""
    SELECT MAX(WeekEndingDate) as QuarterEnd
    FROM {date}
    WHERE FiscalQuarterNbr = (
        SELECT FiscalQuarterNbr
        FROM {date}
        WHERE CalendarDate = DATE_ADD(CURRENT_DATE(), -4)
    )
""")

quarter_end = quarter_end_result.collect()[0]['QuarterEnd']

spark.sql(f"""
INSERT INTO {fact_accountsreceivable}
(
    invoice_balance_type_key,
    reporting_week_ending_date_key,
    source_system_key,
    payor_key,
    client_key,
    office_key,
    start_date_key,
    end_date_key,
    balance,
    age_from_todays_date,
    age_from_quarter_end_date,
    invoice_number,
    episode_in_progress,
    prorated_ar_balance
)
SELECT
    CASE WHEN TOTALDUE > 0 THEN 1 ELSE 2 END     AS invoice_balance_type_key,
    W.WeekEndingDateKey                          AS reporting_week_ending_date_key,
    S.SourceSystemKey                            AS source_system_key,
    P.PayerKey                                   AS payor_key,
    C.ClientKey                                  AS client_key,
    O.OfficeKey                                  AS office_key,
    sd.ServiceDateKey                            AS start_date_key,
    NULL                                         AS end_date_key,
    COALESCE(TOTALDUE, 0)                        AS balance,
    DATEDIFF(CURRENT_DATE(), sd.ServiceDate)     AS age_from_todays_date,
    DATEDIFF('{quarter_end}', sd.ServiceDate) + 4 AS age_from_quarter_end_date,
    INVNO                                        AS invoice_number,
    NULL                                         AS episode_in_progress,
    COALESCE(TOTALDUE, 0)                        AS prorated_ar_balance
FROM {bears_oblist} OB

LEFT JOIN (
    SELECT 
        MAX(ClientKey) AS ClientKey,
        SourceSystemId,
        OfficeNumber
    FROM {client}
    WHERE ClientKey <> 2736498
    GROUP BY SourceSystemId, OfficeNumber
) C
    ON C.SourceSystemId = OB.CLIENTNO
   AND C.OfficeNumber = OB.Office

LEFT JOIN {cubeserviceofficetxnsourcesystem} S
    ON S.SourceSystemName = 'BEARS'

LEFT JOIN {office} O
    ON O.OfficeNumber = OB.Office

LEFT JOIN {payerdimension} P
    ON P.PayerID = OB.BILLTO

LEFT JOIN {cubeserviceofficetxnweekendingdate} W
    ON W.WeekEndingDate = CAST(OB.ReportingWeekendingDate AS DATE)

LEFT JOIN {cubeserviceofficetxnservicedate} sd
    ON sd.ServiceDate = CAST(OB.INVDATE AS DATE)

WHERE OB.INVNO NOT LIKE 'ADV%%'
""")


bears_count = spark.sql(f"""
    SELECT COUNT(*) as count
    FROM {fact_accountsreceivable}
    WHERE Source_System_Key = (
        SELECT SourceSystemKey 
        FROM {cubeserviceofficetxnsourcesystem}
        WHERE SourceSystemName = 'BEARS'
    )
""").collect()[0]['count']


In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW QuarterEndView AS
SELECT MAX(WeekEndingDate) AS QuarterEnd
FROM {date}
WHERE FiscalQuarterNbr = (
    SELECT FiscalQuarterNbr
    FROM {date}
    WHERE CalendarDate = DATE_ADD(CURRENT_DATE(), -4)
)
""")

quarter_end = spark.sql("SELECT QuarterEnd FROM QuarterEndView").collect()[0]['QuarterEnd']
print(f"Quarter End Date: {quarter_end}")

spark.sql(f"""
INSERT INTO {fact_accountsreceivable} (
    invoice_balance_type_key,
    reporting_week_ending_date_key,
    source_system_key,
    payor_key,
    client_key,
    office_key,
    start_date_key,
    end_date_key,
    balance,
    age_from_todays_date,
    age_from_quarter_end_date,
    invoice_number,
    prorated_ar_balance
)
SELECT 
    CASE WHEN clm.Balance > 0 THEN 1 ELSE 2 END as invoice_balance_type_key,
    CAST(REPLACE(CAST(DATE_SUB(CURRENT_DATE(), 4) AS STRING), '-', '') AS INT) as reporting_week_ending_date_key,
    19 as source_system_key,
    pd.PayerKey as payor_key,
    clt.ClientKey as client_key,
    ofc.OfficeKey as office_key,
    CASE 
        WHEN TRY_TO_DATE(clm.ServiceStart, 'MM/dd/yyyy') IS NOT NULL 
        THEN CAST(DATE_FORMAT(TRY_TO_DATE(clm.ServiceStart, 'MM/dd/yyyy'), 'yyyyMMdd') AS INT)
        ELSE NULL
    END as start_date_key,
    CASE 
        WHEN TRY_TO_DATE(clm.ServiceEnd, 'MM/dd/yyyy') IS NOT NULL 
        THEN CAST(DATE_FORMAT(TRY_TO_DATE(clm.ServiceEnd, 'MM/dd/yyyy'), 'yyyyMMdd') AS INT)
        ELSE NULL
    END as end_date_key,
    clm.Balance as balance,
    DATEDIFF(CURRENT_DATE(), TRY_TO_DATE(clm.DateBilled, 'MM/dd/yyyy')) as age_from_todays_date,
    DATEDIFF((SELECT QuarterEnd FROM QuarterEndView), TRY_TO_DATE(clm.DateBilled, 'MM/dd/yyyy')) + 4 as age_from_quarter_end_date,
    clm.ClaimNumber as invoice_number,
    clm.Balance as prorated_ar_balance
FROM {cubhub_alpha_claims} clm
LEFT JOIN {office} ofc
    ON ofc.OfficeNumber = clm.OfficeExternalId
LEFT JOIN (
    SELECT Name, PayerKey
    FROM (
        SELECT 
            *,
            ROW_NUMBER() OVER (PARTITION BY Name ORDER BY PayerKey DESC) as rnb
        FROM {payerdimension}
        WHERE SourceSystemKey = 19
    )
    WHERE rnb = 1
) pd ON pd.Name = clm.PayerName
LEFT JOIN (
    SELECT ClientKey, MedicalRecordNumber
    FROM (
        SELECT 
            ClientKey,
            MedicalRecordNumber,
            ROW_NUMBER() OVER (PARTITION BY MedicalRecordNumber ORDER BY ClientKey DESC) as rnb
        FROM {client}
        WHERE SourceSystem = 'CubHub'
    )
    WHERE rnb = 1
) clt ON clt.MedicalRecordNumber = clm.MedicalRecordNumber
""")

cubhub_fact_count = spark.sql(f"""
    SELECT COUNT(*) as count
    FROM {fact_accountsreceivable}
    WHERE source_system_key = 19
""").collect()[0]['count']

print(f"✓ Total CubHub records in fact table: {cubhub_fact_count}")